# Laboratorio EEG - Bloque 1: procesamiento inicial de una señal EEG

## Objetivo

En este bloque se trabajará con **un solo archivo de ejemplo**. El objetivo es construir la primera parte del pipeline que luego se reutilizará con las señales adquiridas por el grupo:

**lectura -> conversión a µV -> recorte -> preprocesamiento -> análisis en frecuencia -> ventanas**

Cada archivo corresponde a **una sola clase** del protocolo. Por ello, no es necesario crear etiquetas por fases dentro de una misma grabación.

---

## Evaluación en laboratorio: 8 puntos

| Hito | Evidencia esperada | Puntaje |
|---|---|---:|
| 1 | Lectura del archivo, frecuencia de muestreo, canales y eje temporal | 1.0 |
| 2 | Conversión correcta de ADC a µV usando el manual del módulo EEG | 1.5 |
| 3 | Detección de los dos marcadores de `I1` y recorte de la actividad | 1.5 |
| 4 | Preprocesamiento y comparación de la señal antes/después | 1.5 |
| 5 | PSD y análisis de potencia theta, alpha y beta | 1.5 |
| 6 | División de la señal en ventanas para el procesamiento posterior | 1.0 |
|  | **Total** | **8.0** |

#

> El notebook es una guía (de uso referencial). Algunas partes están implementadas y otras deben ser completadas, verificadas o interpretadas por el grupo.

## 1. Importación de librerías

Para este bloque no se utilizarán todavía herramientas de Machine Learning.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import signal

## 2. Configuración

Cambien únicamente los parámetros necesarios para el archivo que estén analizando.

In [ ]:
# Archivo a analizar
DATA_FILE = Path("grupo0_clase0_trial1.txt")

# Canales utilizados en el laboratorio
EEG_COL = "A4"
BUTTON_COL = "I1"

# Clase correspondiente al archivo
# 0 = escucha pasiva
# 1 = 0-back
# 2 = 2-back
TRIAL_CLASS = 0

# Detección del botón
BUTTON_THRESHOLD = 0.5
MIN_EVENT_DISTANCE_S = 0.5

# Margen que se eliminará alrededor de los pulsos del botón
MARGIN_S = 0.5

# Preprocesamiento
LOWCUT_HZ = 2
HIGHCUT_HZ = 30
FILTER_ORDER = 4

# Ventanas que se utilizarán posteriormente para ML
WINDOW_S = 4
OVERLAP = 0.50

# Hito 1 - Lectura del archivo y metadatos [1.0 punto]

La primera tarea es obtener los datos numéricos y la información almacenada en el encabezado de OpenSignals.

La función siguiente realiza la lectura del archivo. Revisen brevemente qué información retorna.

In [ ]:
def read_opensignals_txt(filepath):
    """
    Lee un archivo .txt exportado por OpenSignals.

    Retorna
    -------
    metadata : dict
        Metadatos encontrados en el header.
    df : pandas.DataFrame
        Datos numéricos del registro.
    """
    filepath = Path(filepath)

    with open(filepath, "r", encoding="utf-8") as f:
        lines = f.readlines()

    header_lines = []
    data_start_idx = None

    for i, line in enumerate(lines):
        if line.strip() == "# EndOfHeader":
            data_start_idx = i + 1
            break
        header_lines.append(line.rstrip("\n"))

    if data_start_idx is None:
        raise ValueError("No se encontró '# EndOfHeader' en el archivo.")

    metadata = {}

    for line in header_lines:
        if line.startswith("# {"):
            try:
                metadata = json.loads(line[2:].strip())
            except json.JSONDecodeError:
                print("Advertencia: no se pudo interpretar el JSON del header.")
            break

    columns = None

    if metadata:
        try:
            device_key = list(metadata.keys())[0]
            columns = metadata[device_key].get("column", None)
        except Exception:
            columns = None

    df = pd.read_csv(
        filepath,
        sep="\t",
        comment="#",
        header=None,
        engine="python"
    )

    # Eliminar una posible última columna vacía.
    if df.shape[1] > 0 and df.iloc[:, -1].isna().all():
        df = df.iloc[:, :-1]

    if columns is not None and len(columns) == df.shape[1]:
        df.columns = columns
    else:
        df.columns = [f"col_{i}" for i in range(df.shape[1])]

    return metadata, df

In [ ]:
metadata, df = read_opensignals_txt(DATA_FILE)

print("Columnas disponibles:")
print(df.columns.tolist())

# Extraer frecuencia de muestreo desde el header.
fs = None

if metadata:
    device_key = list(metadata.keys())[0]
    device_info = metadata[device_key]
    fs = device_info.get("sampling rate", None)

if fs is None:
    raise ValueError(
        "No se encontró la frecuencia de muestreo en el header. "
        "Revise el archivo y complete fs manualmente si es necesario."
    )

fs = float(fs)

# Verificar canales necesarios.
for col in [EEG_COL, BUTTON_COL]:
    if col not in df.columns:
        raise ValueError(
            f"No se encontró la columna '{col}'. "
            f"Revise los nombres mostrados anteriormente."
        )

# Eje temporal.
df["time_s"] = np.arange(len(df)) / fs

duration_s = len(df) / fs

print(f"\nFrecuencia de muestreo: {fs:.0f} Hz")
print(f"Número de muestras: {len(df)}")
print(f"Duración del registro: {duration_s:.2f} s")
print(f"Canal EEG: {EEG_COL}")
print(f"Canal botón: {BUTTON_COL}")

display(df.head())

### Verificación del Hito 1

Antes de continuar, comprueben que:

- la frecuencia de muestreo coincide con la configurada durante la adquisición;
- `A4` corresponde al canal EEG;
- `I1` corresponde al botón;
- la duración es razonable para la toma realizada.

# Hito 2 - Conversión de ADC a microvoltios [1.5 puntos]

La columna EEG exportada por OpenSignals contiene valores digitalizados. Para interpretar la amplitud de la señal se convertirá a **microvoltios (µV)**.

Revisen el **manual/datasheet del módulo EEG de BITalino** proporcionado en el laboratorio y utilicen la ecuación de conversión indicada por el fabricante.

### Deben completar

1. Los parámetros necesarios para la conversión.
2. La ecuación dentro de `adc_to_uv()`.
3. Una respuesta breve indicando qué información del manual utilizaron.

In [ ]:
eeg_adc = df[EEG_COL].to_numpy(dtype=float)

# ============================================================
# TODO - Revisar el datasheet del módulo EEG
# ============================================================

ADC_BITS = None
VCC = None
EEG_GAIN = None

def adc_to_uv(adc_values):
    """
    Convertir las cuentas digitales del canal EEG a microvoltios.

    TODO:
    Implementar aquí la ecuación indicada en el manual del módulo EEG.
    """
    raise NotImplementedError(
        "Complete la conversión ADC -> µV utilizando el datasheet."
    )

eeg_uV = adc_to_uv(eeg_adc)

df["eeg_uV"] = eeg_uV

print(f"Mínimo: {np.min(eeg_uV):.2f} µV")
print(f"Máximo: {np.max(eeg_uV):.2f} µV")
print(f"Media: {np.mean(eeg_uV):.2f} µV")

### Respuesta breve - Hito 2

**¿Qué parámetros o ecuación del manual fueron necesarios para convertir la señal a µV?**

> Escriba aquí su respuesta.

## 3. Visualización del registro completo

Antes de filtrar o recortar, observen el registro tal como fue adquirido.

El canal `I1` debe mostrar dos pulsos:

- primer pulso: inicio de la actividad;
- segundo pulso: final de la actividad.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(df["time_s"], df["eeg_uV"], linewidth=0.8)
axes[0].set_title("EEG - registro completo")
axes[0].set_ylabel("Amplitud [µV]")
axes[0].grid(True, alpha=0.3)

axes[1].plot(df["time_s"], df[BUTTON_COL], linewidth=0.8)
axes[1].set_title("Canal de marcador I1")
axes[1].set_xlabel("Tiempo [s]")
axes[1].set_ylabel("I1")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Hito 3 - Detección de marcadores y recorte [1.5 puntos]

La actividad útil se encuentra entre los dos pulsos del botón.

Se utilizará un umbral para convertir `I1` en una señal binaria y luego detectar sus **flancos ascendentes**.

El valor de `BUTTON_THRESHOLD` debe ser revisado a partir de la señal real.

In [ ]:
def detect_rising_edges(
    button_signal,
    fs,
    threshold=0.5,
    min_event_distance_s=0.5
):
    """
    Detecta flancos ascendentes en el canal del botón.
    """
    button_signal = np.asarray(button_signal)

    binary = (button_signal > threshold).astype(int)

    rising_edges = np.where(
        np.diff(binary, prepend=binary[0]) == 1
    )[0]

    # Evitar múltiples detecciones demasiado cercanas.
    min_samples = int(min_event_distance_s * fs)

    filtered = []

    for idx in rising_edges:
        if len(filtered) == 0 or idx - filtered[-1] >= min_samples:
            filtered.append(idx)

    return np.asarray(filtered, dtype=int)


event_indices = detect_rising_edges(
    df[BUTTON_COL].to_numpy(),
    fs=fs,
    threshold=BUTTON_THRESHOLD,
    min_event_distance_s=MIN_EVENT_DISTANCE_S
)

event_times = event_indices / fs

print(f"Eventos detectados: {len(event_indices)}")
print("Índices:", event_indices)
print("Tiempos [s]:", np.round(event_times, 3))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(df["time_s"], df["eeg_uV"], linewidth=0.8)
axes[0].set_ylabel("EEG [µV]")
axes[0].set_title("Verificación de marcadores")
axes[0].grid(True, alpha=0.3)

axes[1].plot(df["time_s"], df[BUTTON_COL], linewidth=0.8)
axes[1].set_ylabel("I1")
axes[1].set_xlabel("Tiempo [s]")
axes[1].grid(True, alpha=0.3)

for t in event_times:
    axes[0].axvline(t, linestyle="--", alpha=0.8)
    axes[1].axvline(t, linestyle="--", alpha=0.8)

plt.tight_layout()
plt.show()

### Selección del intervalo

En una adquisición correcta deberían identificarse dos marcadores principales.

Si se detectan más eventos, revisen el umbral o seleccionen manualmente cuáles corresponden al inicio y final reales.

In [ ]:
# TODO:
# Ajustar estos valores solo si existen eventos extra.
START_EVENT_NUMBER = 0
END_EVENT_NUMBER = 1

if len(event_indices) < 2:
    raise ValueError(
        "Se detectaron menos de dos eventos. Revise BUTTON_THRESHOLD."
    )

start_idx = event_indices[START_EVENT_NUMBER]
end_idx = event_indices[END_EVENT_NUMBER]

# Eliminar un pequeño margen cercano a la pulsación del botón.
margin_samples = int(MARGIN_S * fs)

start_idx_clean = start_idx + margin_samples
end_idx_clean = end_idx - margin_samples

if end_idx_clean <= start_idx_clean:
    raise ValueError("El intervalo seleccionado no es válido.")

eeg_task = df["eeg_uV"].to_numpy()[start_idx_clean:end_idx_clean]
time_task = np.arange(len(eeg_task)) / fs

task_duration_s = len(eeg_task) / fs

print(f"Inicio detectado: {start_idx / fs:.2f} s")
print(f"Final detectado: {end_idx / fs:.2f} s")
print(f"Duración entre marcadores: {(end_idx - start_idx) / fs:.2f} s")
print(f"Duración analizada después del margen: {task_duration_s:.2f} s")

### Respuesta breve - Hito 3

**¿Los marcadores detectados delimitan correctamente la actividad? ¿La duración obtenida es coherente con el protocolo?**

> Escriba aquí su respuesta. Si tuvieron que cambiar el umbral o seleccionar otros eventos, indíquenlo brevemente.

# Hito 4 - Preprocesamiento básico [1.5 puntos]

Para este laboratorio se trabajará principalmente con las bandas theta, alpha y beta.

Se realizará:

1. eliminación de tendencia (`detrend`);
2. filtrado pasa banda entre `LOWCUT_HZ` y `HIGHCUT_HZ`.

El filtro utilizado es Butterworth y se aplica con `sosfiltfilt` para evitar un desfase temporal producido por un filtrado de tipo causal.

In [ ]:
# 1. Eliminar tendencia.
eeg_detrended = signal.detrend(eeg_task)

# 2. Diseñar y aplicar el filtro.
def bandpass_filter(x, fs, lowcut, highcut, order=4):
    sos = signal.butter(
        order,
        [lowcut, highcut],
        btype="bandpass",
        fs=fs,
        output="sos"
    )

    return signal.sosfiltfilt(sos, x)


eeg_filtered = bandpass_filter(
    eeg_detrended,
    fs=fs,
    lowcut=LOWCUT_HZ,
    highcut=HIGHCUT_HZ,
    order=FILTER_ORDER
)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(time_task, eeg_task, linewidth=0.8)
axes[0].set_title("EEG antes del preprocesamiento")
axes[0].set_ylabel("Amplitud [µV]")
axes[0].grid(True, alpha=0.3)

axes[1].plot(time_task, eeg_filtered, linewidth=0.8)
axes[1].set_title(
    f"EEG después del preprocesamiento ({LOWCUT_HZ}-{HIGHCUT_HZ} Hz)"
)
axes[1].set_ylabel("Amplitud [µV]")
axes[1].set_xlabel("Tiempo [s]")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Respuesta breve - Hito 4

**Compare ambas señales. ¿Qué cambios observa después del detrend y del filtrado?**

> Escriba aquí su respuesta. No es necesario realizar todavía una interpretación clínica o fisiológica.

# Hito 5 - Análisis en frecuencia [1.5 puntos]

La forma temporal del EEG no siempre permite identificar fácilmente qué componentes contiene la señal.

Se estimará la **densidad espectral de potencia (PSD)** mediante el método de Welch y luego se calculará la potencia presente en:

- theta: 4-8 Hz;
- alpha: 8-13 Hz;
- beta: 13-30 Hz.

In [ ]:
# PSD mediante Welch.
N_PER_SEG_S = 4

nperseg = min(
    int(N_PER_SEG_S * fs),
    len(eeg_filtered)
)

freqs, psd = signal.welch(
    eeg_filtered,
    fs=fs,
    nperseg=nperseg
)

# Visualizar solo el rango de interés.
freq_mask = (freqs >= 0) & (freqs <= 35)

plt.figure(figsize=(12, 5))
plt.plot(freqs[freq_mask], psd[freq_mask], linewidth=1.2)

plt.axvspan(4, 8, alpha=0.15, label="Theta 4-8 Hz")
plt.axvspan(8, 13, alpha=0.15, label="Alpha 8-13 Hz")
plt.axvspan(13, 30, alpha=0.15, label="Beta 13-30 Hz")

plt.xlabel("Frecuencia [Hz]")
plt.ylabel("PSD [µV²/Hz]")
plt.title("Densidad espectral de potencia - Welch")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def bandpower(freqs, psd, fmin, fmax):
    """
    Integra la PSD dentro de una banda de frecuencia.
    """
    mask = (freqs >= fmin) & (freqs < fmax)

    if np.sum(mask) < 2:
        return np.nan

    return np.trapezoid(
        psd[mask],
        freqs[mask]
    )


BANDS = {
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
}

total_power = bandpower(freqs, psd, 4, 30)

rows = []

for band_name, (fmin, fmax) in BANDS.items():
    absolute_power = bandpower(freqs, psd, fmin, fmax)

    relative_power = (
        absolute_power / total_power
        if total_power > 0
        else np.nan
    )

    rows.append({
        "band": band_name,
        "range_hz": f"{fmin}-{fmax}",
        "absolute_power_uV2": absolute_power,
        "relative_power": relative_power,
        "relative_power_percent": 100 * relative_power,
    })

band_table = pd.DataFrame(rows)

display(band_table)

## Visualización de una banda

Para entender qué significa aislar una banda del EEG, seleccione una de las bandas anteriores y observe su forma temporal.

Puede cambiar `BAND_TO_EXTRACT`.

In [ ]:
BAND_TO_EXTRACT = "theta"

fmin, fmax = BANDS[BAND_TO_EXTRACT]

eeg_band = bandpass_filter(
    eeg_detrended,
    fs=fs,
    lowcut=fmin,
    highcut=fmax,
    order=FILTER_ORDER
)

PLOT_SECONDS = min(10, task_duration_s)
plot_samples = int(PLOT_SECONDS * fs)

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(
    time_task[:plot_samples],
    eeg_filtered[:plot_samples],
    linewidth=0.8
)
axes[0].set_title(
    f"EEG preprocesado ({LOWCUT_HZ}-{HIGHCUT_HZ} Hz)"
)
axes[0].set_ylabel("µV")
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    time_task[:plot_samples],
    eeg_band[:plot_samples],
    linewidth=0.8
)
axes[1].set_title(
    f"Componente {BAND_TO_EXTRACT}: {fmin}-{fmax} Hz"
)
axes[1].set_xlabel("Tiempo [s]")
axes[1].set_ylabel("µV")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Respuesta breve - Hito 5

**¿Cuál de las bandas theta, alpha o beta presenta mayor potencia relativa en este registro?**

> Escriba aquí su respuesta y reporte el valor aproximado.

**¿Este resultado, por sí solo, permite concluir qué estado cognitivo tenía una persona? Explique brevemente.**

> Escriba aquí su respuesta.

# Hito 6 - División en ventanas [1.0 punto]

En el Bloque 2, una toma completa no se utilizará como una única muestra del modelo. La señal se dividirá en ventanas más pequeñas.

En este ejemplo se utilizarán ventanas de `WINDOW_S` segundos con un solapamiento de `OVERLAP`.

**Importante:** el archivo completo ya corresponde a una sola clase. Por ello, no es necesario marcar manualmente una clase diferente para cada ventana.

In [ ]:
def create_windows(
    x,
    fs,
    window_s=4,
    overlap=0.50
):
    """
    Divide una señal 1D en ventanas.

    Retorna
    -------
    windows : np.ndarray
        Matriz [n_ventanas, muestras_por_ventana].
    starts : np.ndarray
        Índice inicial de cada ventana.
    """
    x = np.asarray(x)

    window_samples = int(window_s * fs)

    if not 0 <= overlap < 1:
        raise ValueError("overlap debe estar entre 0 y 1.")

    step_samples = int(
        window_samples * (1 - overlap)
    )

    if step_samples < 1:
        raise ValueError("El paso entre ventanas debe ser >= 1 muestra.")

    windows = []
    starts = []

    for start in range(
        0,
        len(x) - window_samples + 1,
        step_samples
    ):
        end = start + window_samples
        windows.append(x[start:end])
        starts.append(start)

    return np.asarray(windows), np.asarray(starts)


windows, window_starts = create_windows(
    eeg_filtered,
    fs=fs,
    window_s=WINDOW_S,
    overlap=OVERLAP
)

print("Forma del arreglo de ventanas:")
print(windows.shape)

print(f"\nNúmero de ventanas: {windows.shape[0]}")
print(f"Muestras por ventana: {windows.shape[1]}")
print(f"Duración por ventana: {windows.shape[1] / fs:.2f} s")

In [ ]:
# Visualizar la primera ventana.
if len(windows) > 0:
    t_window = np.arange(windows.shape[1]) / fs

    plt.figure(figsize=(12, 4))
    plt.plot(t_window, windows[0], linewidth=0.8)
    plt.xlabel("Tiempo [s]")
    plt.ylabel("EEG [µV]")
    plt.title("Primera ventana del registro")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### Respuesta breve - Hito 6

1. **Si este archivo corresponde a la clase indicada en `TRIAL_CLASS`, qué clase tendrán sus ventanas cuando se construya el dataset del Bloque 2?**

> Escriba aquí su respuesta.

2. **Dos ventanas consecutivas con 50 % de overlap comparten muestras. ¿Por qué podría ser un problema separar aleatoriamente esas ventanas entre entrenamiento y prueba?**

> Escriba aquí su respuesta.

# Opcional - Métricas conductuales del test N-back

El programa del N-back guarda un archivo `*_summary.csv` con información como:

- Balanced Accuracy;
- tiempo de reacción;
- esfuerzo mental reportado por el participante.

Estas métricas **no se utilizarán como características de entrada del clasificador EEG**. Su función principal será servir como información conductual y de control de calidad del ensayo.

Si cuentan con el archivo resumen correspondiente a esta toma, pueden visualizarlo aquí. Esta sección **no suma puntaje en el Bloque 1**.

In [ ]:
# Ejemplo:
# NBACK_SUMMARY_FILE = Path("20260906_S01_2back_summary.csv")

NBACK_SUMMARY_FILE = None

if NBACK_SUMMARY_FILE is not None:
    behavior = pd.read_csv(NBACK_SUMMARY_FILE)

    columns_to_show = [
        col for col in [
            "participant",
            "mode",
            "balanced_accuracy",
            "median_hit_rt_ms",
            "mental_effort_1_to_7"
        ]
        if col in behavior.columns
    ]

    display(behavior[columns_to_show])

else:
    print(
        "Sección opcional. Asigne una ruta a NBACK_SUMMARY_FILE "
        "si desea cargar las métricas del test."
    )

## Qué se reutilizará en el Bloque 2

El código desarrollado aquí servirá como base para procesar todos los archivos del dataset.

En el Bloque 2 también podrán integrar las métricas conductuales del N-back como **control de calidad** y para comparar el comportamiento entre condiciones.

# Entrega del Bloque 1

Antes de entregar, verifiquen que el notebook cumpla con los criterios presentados al inicio de este notebook para el puntaje completo.

Entreguen el archivo en formato `.ipynb` o `.html`.